# 03 Use the trained model

Predicts void, wall, filament and halo for every cell of a simulation box
with the published DeepVoid model, without training anything. The model was
trained on TNG300-3 subhalos at mean separations of 3 to $10\,h^{-1}\,\mathrm{Mpc}$
([doi:10.5281/zenodo.24261832](https://doi.org/10.5281/zenodo.24261832)).

Here it runs on the published TNG300-3 grids
([doi:10.5281/zenodo.24233069](https://doi.org/10.5281/zenodo.24233069)) and
is scored on the slab of the box it never saw in training. Section 4 shows
how to run it on your own box.

Downloads: the model (113 MB) and two grids (the subhalos at
$\lambda = 10\,h^{-1}\,\mathrm{Mpc}$ and the T-web labels). Use a GPU runtime.

In [ ]:
# Get the code
!rm -rf /content/deepvoid && git clone -q https://github.com/skumagai3/deepvoid.git /content/deepvoid
!pip -q install /content/deepvoid matplotlib

## 1. Download the model and the data

In [ ]:
from deepvoid import data, pretrained

model = pretrained.load(folder='/content/deepvoid_models')
catalog = data.fetch(['tng300-3/subhalos/L10', 'tng300-3/mask'], '/content/deepvoid_data')
print(model.info['name'], '| trained spacings:', model.spacings,
      '| cell size:', model.info['cell_size'], 'h^-1 Mpc')

## 2. Predict the whole box

`predict` returns a score for each class in every cell; the class with the
highest score is the prediction. The input is built exactly as in training:
divided by its mean, a smoothed copy added as a second channel, and both
scaled with the statistics saved with the model.

In [ ]:
tracers = catalog.load('tng300-3/subhalos/L10')
truth = catalog.load('tng300-3/mask').data
scores = model.predict(tracers.data, box_size=tracers.box_size, spacing=10.0)
pred = scores.argmax(-1)
print(tracers.data.shape, '->', pred.shape)

## 3. Score it where it was not trained

The box was split into slabs along x before training (70% train, 15%
validation, 15% test, with buffers between them). Only the test slab was
never used for training or for choosing the model, so that is where the
scores mean something; the whole-box score includes the training slab.

In [ ]:
import yaml
from deepvoid.evaluate import confusion_matrix, scores as score_cm
from deepvoid.split import slab_split

cfg = yaml.safe_load(open(model.folder / 'config.yaml'))
test = slab_split(truth.shape[0], **cfg['data']['split'])['test']
sel = test.mask(truth.shape[0])
for name, m in (('test slab', sel), ('whole box (includes training data)', None)):
    cm = confusion_matrix(truth[sel], pred[sel]) if m is not None else confusion_matrix(truth, pred)
    s = score_cm(cm)
    print(f"{name:36s} four-class MCC {s['mcc']:.3f}   void MCC {s['void_mcc']:.3f}   "
          f"void F1 {s['void_f1']:.3f}")

In [ ]:
import matplotlib.pyplot as plt
from deepvoid.figures import prediction_figure

i = (test.start + test.stop) // 2          # a slice through the middle of the test slab
x = model.inputs(tracers.data, tracers.box_size, 10.0)
fig = prediction_figure(x, truth, scores, index=i, axis=0, box_size=tracers.box_size,
                        title=f'Test slab, x = {i * tracers.box_size / truth.shape[0]:.0f} h^-1 Mpc, '
                              'subhalos at 10 h^-1 Mpc mean separation',
                        input_label='smoothed subhalo input')
plt.show()

## 4. Your own box

The model expects a periodic box gridded like its training data:

- cells of $0.40\,h^{-1}\,\mathrm{Mpc}$ (within 5%); other cell sizes are refused;
- the mass of the most massive tracers, enough of them for a mean separation
  between 3 and $10\,h^{-1}\,\mathrm{Mpc}$, assigned by cloud-in-cell.

`Project.add_tracers` makes such a grid from positions and masses. For example,
with `pos` (N, 3) in $h^{-1}\,\mathrm{Mpc}$ and `mass` (N,) for a box of side `L`:

```python
from deepvoid import Project
project = Project('/content/my_project')
n = round(L / model.info['cell_size'])
grid = project.add_tracers('halos/L7', pos, box_size=L, n=n, masses=mass, spacing=7.0)
classes = model.predict(grid.data, box_size=L, spacing=7.0).argmax(-1)
```

The model was tested on TNG300-3 and, without retraining, on Bolshoi halos
(four-class MCC within 0.012 of TNG at every spacing). It was trained in real
space; on redshift-space positions its scores drop sharply.